# 02 – Yapılandırılmış çıktı: Talepleri sınıflandırma

*Hafta 1 notebook'u.* Bankaya gelen (sentetik) talepleri `data/talepler.csv` dosyasından okuyup yerel modele sınıflandıracağız.
Önce modelden kategoriyi **serbest metin** olarak isteyeceğiz ve neden işe yaramadığını göreceğiz.
Sonra aynı işi bir **şema** ile yapıp her cevabın programın kullanabileceği biçimde gelmesini sağlayacağız.

> Notebook'un sağ üst köşesinde projenin Python ortamı seçili olmalı (Dev Container'da *Python 3*, kendi bilgisayarınızda `.venv`).

## Bu notebook neden var?

Kısaca: **modelin cevabını bir programın güvenle kullanabileceği hale getirmek.**

### Çözdüğü sorun
Isınma notebook'unda (`01_isinma.ipynb`) modelin cevabını biz okuduk. Bir insan için uzun bir paragraf da,
kalın yazılmış bir kelime de sorun değil, ne denmek istendiğini anlarız.

Bankanın asistanında ise cevabı okuyan bir **program**. Program cevaba bakıp karar verecek: bu talep kart birimine mi
gidecek, insana mı aktarılacak, müşteriye cevap mı yazılacak? Bunun için cevabın her seferinde **aynı biçimde** gelmesi gerekir.
2. bölümde bunun neden kendiliğinden olmadığını göreceksiniz.

### Çözüm
3. bölümde modele bir **form** veriyoruz: "Kategori şu 6 değerden biri olacak, yanına bir cümle gerekçe yazacaksın."
Böylece her cevap bu formda gelir ve program doğrudan `sonuc.kategori` diyerek okuyabilir.

### Neden 5 hafta boyunca önemli
Bu notebook'taki yöntem projenin her yerinde kullanılıyor:
- **`cevapla()` fonksiyonu** (`src/asistan/tek_cagri.py`) tam olarak bu yöntemle çalışıyor. Bu notebook onun içini adım adım gösteriyor.
- **Değerlendirme** (`eval/calistir.py`) modelin cevabını beklenen etiketle karşılaştırıyor. Bu ancak cevap düzgün biçimde gelirse mümkün.
- **Hafta 2'deki araçlar** da aynı mekanizmayla çalışacak: model "müşteri bilgisini getir" gibi bir aracı çağırırken de bir forma uygun cevap verir.

### İkinci, daha ince ders
Notebook'un sonunda şunu da kendiniz göreceksiniz: **biçim düzeldi diye karar doğru olmayabilir.**
Bu yüzden haftanın geri kalanında önce doğruluğu ölçeceğiz (Görev 2), hataları inceleyeceğiz (Görev 3),
sonra sistem mesajını iyileştireceğiz (Görev 4).

> **Tek cümleyle:** `01_isinma` modelle nasıl konuşulacağını gösterir, bu notebook ise modelin cevabının bir programın parçası olarak nasıl kullanılacağını.

In [ ]:
# Proje kökünü bul ve src/ klasörünü import yoluna ekle
import sys
from pathlib import Path

PROJE_KOKU = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kontrol.py").exists())
sys.path.insert(0, str(PROJE_KOKU / "src"))

from asistan.ayarlar import ayar, model_getir

print("Yerel model:", ayar("OLLAMA_MODEL"), "@", ayar("OLLAMA_URL"))

### Satır satır açıklama

**`# Proje kökünü bul ve src/ klasörünü import yoluna ekle`**
`#` ile başlayan satır **yorumdur**. Python bu satırı çalıştırmaz, sadece okuyana bilgi verir.

**`import sys`**
Python'la birlikte gelen `sys` modülünü yükler (ayrıca kurmak gerekmez). Bu modülle Python'un kendi ayarlarına erişiriz. Bize lazım olan `sys.path` listesi: Python, `import` gördüğünde modülleri bu listedeki klasörlerde arar.

**`from pathlib import Path`**
`pathlib` modülünden `Path` aracını alır. `Path` dosya ve klasör yollarını metin olarak değil, **nesne** olarak tutar:
- Yolları `/` işaretiyle birleştirebiliriz: `klasor / "dosya.py"`.
- Windows'taki `\` ile Mac'teki `/` farkını kendisi halleder.

**`PROJE_KOKU = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kontrol.py").exists())`**
En karmaşık satır bu. İçten dışa doğru:

| Parça | Ne yapar | Örnek değer |
|---|---|---|
| `Path.cwd()` | Şu an çalışılan klasör (*current working directory*). Notebook kendi klasöründe çalışır. | `.../softtech_agentic/haftalar/hafta-1` |
| `Path.cwd().parents` | Bu klasörün bütün üst klasörleri, yakından uzağa | `.../haftalar`, `.../softtech_agentic`, `.../Desktop`, ... `/` |
| `[Path.cwd(), *Path.cwd().parents]` | Hepsini tek listede toplar. `*` işareti "`parents` içindekileri tek tek listeye dök" demek. | `[hafta-1, haftalar, softtech_agentic, Desktop, ...]` |
| `p / "kontrol.py"` | Listedeki her klasörün (`p`) sonuna `kontrol.py` ekler | `.../hafta-1/kontrol.py` |
| `.exists()` | O dosya gerçekten var mı? `True` / `False` | |
| `p for p in [...] if ...` | Klasörleri sırayla dolaşır, sadece içinde `kontrol.py` olanları seçer | |
| `next(...)` | Seçilenlerden **ilkini** alır ve aramayı bitirir | |

Arama şöyle ilerler:
- `hafta-1/kontrol.py` var mı? **Yok**, geç.
- `haftalar/kontrol.py` var mı? **Yok**, geç.
- `softtech_agentic/kontrol.py` var mı? **Var**, bunu seç.

Sonuç `PROJE_KOKU` değişkenine kaydedilir: `.../softtech_agentic`

*Neden bu kadar uğraşıyoruz?* Proje Masaüstünde, başka bir sürücüde ya da Dev Container'da (`/workspace`) olabilir. Yolu koda elle yazmak yerine `kontrol.py` dosyasını "işaret taşı" olarak kullanıyoruz. Böylece kod her bilgisayarda çalışır.

**`sys.path.insert(0, str(PROJE_KOKU / "src"))`**
- `PROJE_KOKU / "src"` → proje kökünün sonuna `src` ekler: `.../softtech_agentic/src`
- `str(...)` → yolu düz metne çevirir. `sys.path` metinlerden oluşan bir liste olduğu için gerekli.
- `sys.path.insert(0, ...)` → bu yolu listenin **0. sırasına**, yani en başına ekler.

*Neden?* `asistan` klasörü `src/` içinde duruyor ve Python normalde oraya bakmaz. `src/` yolunu ekleyince `import asistan` çalışır. En başa eklememizin sebebi, aynı adlı başka bir paket varsa önce bizimkinin bulunması.

**`from asistan.ayarlar import ayar, model_getir`**
- `asistan.ayarlar` → `src/asistan/ayarlar.py` dosyası. Nokta, klasör ayracı gibi düşünülebilir.
- `import ayar, model_getir` → o dosyadaki iki fonksiyonu bu notebook'ta kullanılabilir yapar.

Bu satır çalışınca arka planda şunlar olur:
1. Python önce `src/asistan/__init__.py` dosyasını çalıştırır. Bir klasörden ilk kez bir şey alındığında hep bu dosya çalışır.
2. `__init__.py` de `asistan.py` dosyasını yükler. O dosya `pydantic` paketini ister. Notebook yanlış ortamda (kernel) açıksa `ModuleNotFoundError` hatası **tam burada** çıkar. Çözümü: sağ üstten `.venv` kernel'ini seçmek.
3. `ayarlar.py` çalışır ve `.env` dosyasını okur.

Alınan fonksiyonlar:
- `ayar("OLLAMA_MODEL")` → `.env` dosyasındaki değeri döner. Değer boşsa anlaşılır bir hata verir.
- `model_getir("yerel")` → kullanıma hazır bir model nesnesi döner (2. bölümde kullanılıyor).

**`print("Yerel model:", ayar("OLLAMA_MODEL"), "@", ayar("OLLAMA_URL"))`**
- `ayar("OLLAMA_MODEL")` → model adı: `qwen3.5:4b`
- `ayar("OLLAMA_URL")` → Ollama'nın adresi: `http://localhost:11434`
- `print(...)` → virgülle ayrılan parçaları aralarına boşluk koyarak yan yana yazar.

Çıktı:
```
Yerel model: qwen3.5:4b @ http://localhost:11434
```
Amaç kontrol: ayarlar doğru okunuyor mu, hangi modelle hangi adrese bağlanacağız? `.env` eksikse hata erkenden, burada ortaya çıkar.

> **Özet:** Proje klasörünü bul, `src/` klasörünü Python'a göster, ayar fonksiyonlarını yükle ve ayarların doğru olduğunu ekranda gör.

## 1. Gelen talepler

`data/talepler.csv` bankaya gelen 40 sentetik talebi içerir. Dikkat: bazı taleplerde TCKN, kart numarası, IBAN gibi hassas veriler var.
Bu veriler sahte ama **ileriki haftalarda bunları modele göndermeden önce maskelemeyi** öğreneceğiz.

In [ ]:
import csv

with open(PROJE_KOKU / "data" / "talepler.csv", encoding="utf-8") as f:
    talepler = list(csv.DictReader(f))

print(len(talepler), "talep")
for t in talepler[:10]:
    print(f"{t['talep_id']} [{t['kanal']:<13}] {t['talep']}")

### Satır satır açıklama

**`import csv`**
Python'la gelen `csv` modülünü yükler. CSV "virgülle ayrılmış değerler" demektir: her satır bir kayıt, sütunlar virgülle ayrılmış. Excel tablosunun düz metin hali gibi düşünebilirsiniz.

**`with open(PROJE_KOKU / "data" / "talepler.csv", encoding="utf-8") as f:`**
- `PROJE_KOKU / "data" / "talepler.csv"` → ilk hücrede bulduğumuz proje kökünden başlayan tam yol.
- `open(...)` → dosyayı **okumak için** açar.
- `encoding="utf-8"` → karakter kodlaması. Bu olmazsa özellikle Windows'ta "ş, ğ, ı, ö, ü, ç" bozuk görünür.
- `with ... as f:` → dosyayı `f` adıyla kullanırız. Blok bitince **dosyayı kendiliğinden kapatır**, hata olsa bile. Dosya açmanın güvenli yolu budur.

**`talepler = list(csv.DictReader(f))`**
- `csv.DictReader(f)` → dosyayı satır satır okur. İlk satırı (`talep_id,musteri_no,kanal,tarih,talep`) **sütun adları** olarak kabul eder ve her satırı bir **sözlüğe** çevirir:
  ```python
  {"talep_id": "T0001", "musteri_no": "M1004", "kanal": "web", "tarih": "2026-09-29",
   "talep": "Merhaba, kredi kartı ekstremin kesim tarihini nereden görebilirim?"}
  ```
- `list(...)` → `DictReader` satırları tek tek ve istendikçe verir. `list` hepsini bir kerede okuyup listeye koyar. Bunu dosya kapanmadan, yani `with` bloğunun içinde yapmamız gerekiyor.
- `talepler` → artık sözlüklerden oluşan bir liste.

Bir sonraki satırda girinti bitiyor: `with` bloğu sona erdi ve dosya kapandı. `talepler` listesi ise hafızada duruyor.

**`print(len(talepler), "talep")`**
`len(...)` listedeki eleman sayısını verir. Çıktı: `40 talep`

**`for t in talepler[:10]:`**
- `talepler[:10]` → listenin **ilk 10 elemanı**. Hepsini yazdırsak ekran dolar.
- Her turda sıradaki talep (bir sözlük) `t` değişkenine konur.

**`print(f"{t['talep_id']} [{t['kanal']:<13}] {t['talep']}")`**
- `t['talep_id']` → sözlükten `talep_id` alanını okur. Köşeli parantez içinde **tek tırnak** var, çünkü dışarıda f-string'in çift tırnağı kullanılıyor.
- `{t['kanal']:<13}` → kanal adını bir **biçimle** yazar:
  - `:` → "biçim kuralı geliyor"
  - `<` → sola yasla
  - `13` → 13 karakterlik alan kapla, kısa kelimelerin sonuna boşluk ekle
  - Böylece `web` ile `cagri_merkezi` aynı genişlikte görünür ve sütunlar hizalı durur.
- `[` ve `]` → kanal adını köşeli parantez içinde göstermek için düz metin.
- `{t['talep']}` → müşterinin talep metni.

Örnek çıktı:
```
T0001 [web          ] Merhaba, kredi kartı ekstremin kesim tarihini nereden görebilirim?
T0003 [web          ] Kartımı kaybettim, numarası 9999 0000 0000 0005. Lütfen hemen kapatın!
T0007 [cagri_merkezi] Hesabımdan hesap işletim ücreti kesilmiş, bunu kabul etmiyorum, iadesini istiyorum.
T0008 [eposta       ] Ben Emre Örnek, TCKN 99999000015. IBAN'ım TR00 9999 ...
```

> **Özet:** Sentetik müşteri talepleri dosyadan okunuyor ve ilk 10 tanesi düzenli biçimde gösteriliyor. T0003 ve T0008'de kart numarası, TCKN ve IBAN var (hepsi sahte). Bunları ileriki haftalarda modele göndermeden önce maskelemeyi göreceğiz.

## 2. Serbest metinle sınıflandırma

Modelden bir etiket istediğimizde cevap serbest metin olarak gelirse, programın bu metinden etiketi *ayrıştırması* gerekir. Bakalım ne oluyor:

In [ ]:
model = model_getir("yerel")

serbest_sistem = "Talebi şu kategorilerden birine ata: kart, kredi, hesap, dijital_kanal, guvenlik, diger."

for t in talepler[:5]:
    cevap = model.invoke([("system", serbest_sistem), ("human", t["talep"])])
    print(repr(cevap.text[:80]))

### Satır satır açıklama

**`model = model_getir("yerel")`**
- Yerel modeli (Ollama'daki qwen3.5:4b) hazırlar ve `model` değişkenine kaydeder.
- Bu satır modele henüz bir şey sormaz, sadece bağlantı ayarlarını hazırlar.

**`serbest_sistem = "Talebi şu kategorilerden birine ata: kart, kredi, hesap, dijital_kanal, guvenlik, diger."`**
- Modele vereceğimiz **talimatı** bir değişkene kaydeder. Bu talimat bir *sistem mesajı* olarak gönderilecek (aşağıda).
- Modele 6 kategori veriyoruz, ama **cevabın biçimi için hiçbir kural koymuyoruz**: "sadece kelimeyi yaz", "küçük harf kullan" gibi bir şey demiyoruz. Bilerek böyle bırakıldı, çünkü bu bölümün amacı sorunu göstermek.

**`for t in talepler[:5]:`**
İlk 5 talebi sırayla dolaşır. Her turda sıradaki talep (bir sözlük) `t` değişkenine konur.

**`cevap = model.invoke([("system", serbest_sistem), ("human", t["talep"])])`**
- Isınma notebook'unda `.invoke()` fonksiyonuna sadece soruyu vermiştik. Bu sefer bir **mesaj listesi** veriyoruz. Her mesaj bir `(rol, metin)` ikilisi (Python'da buna *tuple* denir):
  - `("system", serbest_sistem)` → **sistem mesajı**: modelin görevi ve kuralları. Müşteri bunu görmez, sadece modelin davranışını yönlendirir.
  - `("human", t["talep"])` → **kullanıcı mesajı**: müşterinin talebi. `t["talep"]` sözlükten talep metnini alır.
- Model ikisini birlikte okur ve talimata göre cevap verir.
- Bu satır döngünün içinde olduğu için **5 kez** çalışır, her seferinde farklı bir taleple.

**`print(repr(cevap.text[:80]))`**
- `cevap.text` → modelin cevap metni.
- `[:80]` → ilk 80 karakter.
- `repr(...)` → metni **olduğu gibi** yazar: tırnaklarla birlikte ve satır sonu (`\n`) gibi gizli karakterler görünür halde. Normal `print` ile bu farklar fark edilmez.

Örnek çıktı:
```
'Bu sorunuz **kredi** kategorisine aittir.\n\nKredi kartınızın kesim tarihini (baki'
'diger'
'Bu durum için en uygun kategori: **guvenlik**\n\nNedeni: Kartınızın kaybedilmesi v'
'Konut kredinizi erken kapatırken ödeyeceğiniz ücret miktarı, bankanızın politika'
'dijital_kanal'
```
(Sizin çıktınız biraz farklı olabilir: yerel model aynı talebe her seferinde birebir aynı cevabı vermeyebilir.)

> **Özet:** Cevapların biçimi her seferinde farklı: bazen tek kelime (`'diger'`), bazen kalın yazılmış kelimeli uzun bir cümle (`**kredi**`). 4. talepte model **hiç kategori söylemedi**, talimatı unutup doğrudan müşterinin sorusunu cevaplamaya başladı. Bir program `if cevap == "kart":` diye kontrol etse çoğu durumda yanılır. Bankacılık gibi güvenilirlik isteyen bir alanda bu kabul edilemez.

## 3. Şemayla sınıflandırma

Bazen sadece etiket, bazen açıklamalı cümle, bazen büyük harf... Bu çıktıyla güvenilir bir sistem kurulamaz.

**Çözüm:** İstediğimiz çıktının şemasını Pydantic ile tanımlayıp `with_structured_output()` kullanmak. Model artık şemaya uygun çıktı üretiyor.

> **Not:** Yerel modelde `method="function_calling"` kullanıyoruz: şema modele bir *araç* olarak verilir ve model o aracı çağırır. Claude'da `method="json_schema"` kullanılır; orada API çıktının şemaya uymasını **zorunlu** kılar. Mac'teki Ollama (MLX motoru) `json_schema`'yı her zaman desteklemediği için yerelde araç yöntemini seçtik.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Etiket(BaseModel):
    kategori: Literal["kart", "kredi", "hesap", "dijital_kanal", "guvenlik", "diger"]
    gerekce: str = Field(description="Tek cümlelik gerekçe")

yapili_model = model.with_structured_output(Etiket, method="function_calling")

for t in talepler[:5]:
    sonuc = yapili_model.invoke([("system", "Müşteri talebini sınıflandır."), ("human", t["talep"])])
    print(f"{sonuc.kategori:<14} | {sonuc.gerekce}")

### Satır satır açıklama

**`from typing import Literal`**
Python'la gelen `typing` modülünden `Literal` aracını alır. `Literal` "bu değer **sadece şu seçeneklerden biri** olabilir" demek için kullanılır.

**`from pydantic import BaseModel, Field`**
`pydantic` kütüphanesinden iki araç alır:
- `BaseModel` → verinin **şeklini** (hangi alanlar var, türleri ne) tanımlamak için temel sınıf. Pydantic, gelen verinin bu şekle uyup uymadığını **otomatik kontrol eder**.
- `Field` → bir alana ek bilgi, örneğin açıklama eklemek için.

**`class Etiket(BaseModel):`**
- `class Etiket` → `Etiket` adında yeni bir veri tipi tanımlıyoruz. Bunu bir **form şablonu** gibi düşünün: hangi kutular var, her kutuya ne yazılabilir.
- `(BaseModel)` → bu şablon Pydantic'in özelliklerini taşısın: tür kontrolü, şemaya dönüşebilme.

**`kategori: Literal["kart", "kredi", "hesap", "dijital_kanal", "guvenlik", "diger"]`**
- `kategori` → formdaki ilk alanın adı. `:` sonrası alanın **türü**.
- `Literal[...]` → alan **sadece bu 6 değerden biri** olabilir. `"Kart"` veya `"kredi kartı"` kabul edilmez.
- Bu 6 değer, `eval/test_seti.csv` dosyasındaki `beklenen_kategori` sütunuyla birebir aynı.

**`gerekce: str = Field(description="Tek cümlelik gerekçe")`**
- `gerekce` → ikinci alan. `str` → türü metin, serbest yazı olabilir.
- `Field(description=...)` → alana bir **açıklama** ekler. Bu açıklama modele de gönderilir, model bu alana ne yazacağını buradan öğrenir.

Gerekçe alanının iki faydası var:
1. Modelin neden o kategoriyi seçtiğini görürüz, bu da hata ayıklamayı kolaylaştırır.
2. Modelden gerekçe istemek bazen kararın kalitesini artırır.

**`yapili_model = model.with_structured_output(Etiket, method="function_calling")`**
Hücrenin en önemli satırı bu.
- `.with_structured_output(Etiket, ...)` → modeli **sarar** ve yeni bir model nesnesi döner. Bu yeni nesne:
  1. `Etiket` şemasını modele iletir ("cevabını bu formda ver").
  2. Gelen cevabı bir `Etiket` nesnesine çevirir.
  3. Cevap şemaya uymazsa (ör. geçersiz bir kategori) bunu yakalar.
- `method="function_calling"` → şemanın modele **nasıl** iletileceği:
  - Şema modele bir **araç** (fonksiyon) olarak tanıtılır: "Elinde `Etiket(kategori, gerekce)` diye bir fonksiyon var, onu çağır."
  - Model de cevabı serbest metin olarak değil, bu fonksiyonun parametreleri olarak verir.
  - Yerel modelde bu yöntemi kullanıyoruz, çünkü Mac'teki Ollama `json_schema` yöntemini her zaman desteklemiyor. Claude'da `json_schema` kullanılır ve orada şemaya uyum API tarafından **zorunlu** kılınır.
- Orijinal `model` değişmez, hâlâ serbest metin döner. Sarılmış model `yapili_model` değişkenine kaydedilir.

**`for t in talepler[:5]:`**
Önceki hücredeki **aynı 5 talebi** dolaşıyoruz. Böylece iki yöntemi doğrudan karşılaştırabiliyoruz.

**`sonuc = yapili_model.invoke([("system", "Müşteri talebini sınıflandır."), ("human", t["talep"])])`**
- Bu sefer `model` yerine `yapili_model` çağrılıyor.
- Sistem mesajı çok kısa. Kategori listesini yazmaya gerek yok, çünkü model geçerli kategorileri şemadan öğreniyor.
- **Büyük fark:** `sonuc` artık bir `AIMessage` değil, bir **`Etiket` nesnesi**.

**`print(f"{sonuc.kategori:<14} | {sonuc.gerekce}")`**
- `sonuc.kategori` → alana **doğrudan** erişiyoruz. Metni ayrıştırmaya gerek yok, değer her zaman 6 geçerli değerden biri.
- `:<14` → 14 karakterlik alana sola yaslı yaz. En uzun kategori `dijital_kanal` 13 karakter.
- `|` → iki sütunu ayırmak için düz metin.

Örnek çıktı:
```
kredi          | Müşteri kredi kartı ekstresi kesim tarihini sorgulamak istiyor.
kredi          | Müşterinin kartından yapılan 3677,79 TL'lik iki adet çekimden birini iade etmesi isteniyor.
guvenlik       | Müşterinin kart numarasını kaybetmesi ve acil olarak kapatılması talebi.
kredi          | Erken kapatma ücreti sorgusu
dijital_kanal  | Müşteri mobil uygulamayı kullanamıyor ve şifre sıfırlama işlemi talep ediyor.
```
(Sizin çıktınız biraz farklı olabilir.)

> **Özet:** Biçim sorunu çözüldü: her cevap aynı şekilde ve programın doğrudan kullanabileceği biçimde geliyor.
>
> **Ama dikkat, biçimin doğru olması kararın doğru olduğu anlamına gelmez.** Çıktıya bakın: T0001 (kredi kartı ekstresi) ve T0002 (karttan çift çekim) **kart** konuları, ama model ikisine de `kredi` dedi. Büyük ihtimalle talepteki "kredi kartı" kelimesi modeli yanıltıyor. Yerel model kararsız da kalabiliyor: başka bir denemede T0005'e (mobil uygulama şifresi) `dijital_kanal` yerine `guvenlik` dedi. Yapılandırılmış çıktı *biçimi* garanti eder, *doğruluğu* değil. Doğruluğu artırmak için kategori sınırlarını sistem mesajında açıkça tarif etmek gerekir (bkz. `GOREVLER.md`), doğruluğu ölçmek için de `eval/calistir.py` kullanılır.

## Tartışma

Aşağıdaki soruları düşünün. Cevaplarınızı bu hücrenin altına yeni bir markdown hücresi ekleyerek yazın (Görev 1 teslimi).

1. **Ayrıştırma:** 2. bölümdeki serbest metin cevaplarından kategoriyi çıkaran bir kod yazmaya çalışsaydınız, hangi cevaplarda yanılırdı?
2. **Biçim ve doğruluk:** 3. bölümde 5 talepten kaçı doğru sınıflandı? Talepleri kendiniz okuyup karar verin. Biçimin doğru olması kararın da doğru olduğu anlamına gelir mi?
3. **Gerekçe:** Modelin yanlış sınıflandırdığı bir talepte `gerekce` alanı ne söylüyor? Bu bilgi hatayı düzeltmenize nasıl yardım eder?
4. **Değişiklik:** Banka yeni bir `yatirim` kategorisi eklemek isteseydi, kodda nereyi değiştirmeniz gerekirdi? Serbest metin yönteminde aynı değişiklik ne kadar zor olurdu?